# C11-neural-training — Session 4: PyTorch Autograd and Optimizers

*One 90-minute session. Prerequisites: C6's tensors, `nn.Module`,
`nn.Parameter`, `requires_grad`, and inspection; Sessions 1–3 supply the
loss, backward pass, and training lifecycle.*

**Learning contract.** We will identify leaves and nonleaves, compare one
manual gradient with autograd, expose gradient accumulation, use the exact
optimizer ordering, inspect optimizer state, and certify a deterministic
CPU-small training loop.


In [ ]:
import torch
from torch import nn

SEED = 20260804
ATOL = 1e-9
RTOL = 1e-7
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
torch.use_deterministic_algorithms(True)

## 1. Dynamic graphs, leaves, and `requires_grad`

A tensor with `requires_grad=True` asks autograd to record operations needed
for derivatives. A **leaf** is a graph input created directly by the user,
including each `nn.Parameter`. An operation result is a **nonleaf** and
usually has `grad_fn`.

After `loss.backward()`, gradients accumulate by default in the `.grad`
fields of leaf parameters. A nonleaf's gradient is used during propagation
but is not retained unless `retain_grad()` is requested before backward.

Shapes do not change: if $W$ has shape $(H,D)$, then `W.grad` has shape
$(H,D)$.

**Checkpoint 1A.** If `w` is a leaf requiring gradients and `z = w.square().sum()`,
which is a leaf and what is `z.grad_fn` evidence of?

**Checkpoint 1B.** Does `requires_grad=True` cause a parameter to move by
itself?


In [ ]:
w = torch.tensor([2.0, -3.0], requires_grad=True)
z = w.square().sum()
print("w leaf:", w.is_leaf, "| z leaf:", z.is_leaf, "| z grad_fn:", type(z.grad_fn).__name__)
assert w.is_leaf and not z.is_leaf
z.backward()
print("w.grad:", w.grad)
assert torch.allclose(w.grad, torch.tensor([4.0, -6.0]), atol=ATOL, rtol=RTOL)

## 2. Hand gradient versus autograd

For $L=\operatorname{mean}(Xw-y)^2$, with $X(N,D)$, $w(D)$, and $y(N)$,

$$\nabla_w L=\frac{2}{N}X^\top(Xw-y),$$

shape $(D)$. We compute the same value through autograd. This checks the
mathematical contract without asking either path to reproduce trained weights.

**Checkpoint 2A.** Why does $X^\top(Xw-y)$ have shape $(D)$?

**Checkpoint 2B.** If the loss were a sum rather than a mean, which factor
would disappear?


In [ ]:
X_check = torch.tensor([[1.0, 2.0], [-1.0, 3.0], [2.0, 0.5]])
y_check = torch.tensor([1.0, -2.0, 0.5])
w_check = torch.tensor([0.25, -0.5], requires_grad=True)
residual = X_check @ w_check - y_check
loss_check = residual.square().mean()
manual = (2.0 / X_check.shape[0]) * X_check.T @ residual.detach()
loss_check.backward()
print("manual:", manual, "| autograd:", w_check.grad)
assert torch.allclose(w_check.grad, manual, atol=ATOL, rtol=RTOL)

## 3. Gradient accumulation is visible state

Calling `backward()` adds into an existing leaf `.grad`; it does not replace
it. Two backward passes on two newly built graphs therefore produce the sum of
the two gradients. This is useful for deliberate gradient accumulation across
micro-batches, but it is a common accidental bug.

`optimizer.zero_grad(set_to_none=True)` clears the old state. With
`set_to_none=True`, a parameter not used by the next loss remains `None`,
which can expose a disconnected parameter; a zero tensor would hide that
distinction.

**Checkpoint 3A.** If one backward produces gradient $g$, what appears after
a second identical backward without clearing?

**Checkpoint 3B.** Why must a loop clear gradients *before* the forward/loss/
backward sequence whose update they should control?


In [ ]:
u = torch.tensor([1.5, -2.0], requires_grad=True)
(u.square().sum()).backward()
once = u.grad.clone()
(u.square().sum()).backward()
twice = u.grad.clone()
print("once:", once, "| twice:", twice)
assert torch.allclose(twice, 2 * once, atol=ATOL, rtol=RTOL)
u.grad = None
assert u.grad is None

## 4. Optimizer ordering is a state machine

The required training order is:

1. `optimizer.zero_grad(set_to_none=True)`;
2. `logits = model(X)`;
3. `loss = criterion(logits, y)`;
4. `loss.backward()`;
5. `optimizer.step()`.

`step()` reads current `.grad` fields and mutates parameters. It does not
run forward or backward. Calling it before `backward()` either does nothing
(first step with `None` gradients) or reuses stale gradients. Clearing after
backward but before step discards the evidence needed for the update.

**Checkpoint 4A.** What exact bug results from
`forward → backward → zero_grad → step`?

**Checkpoint 4B.** Why should evaluation not call `optimizer.step()` even
inside `torch.no_grad()`?


In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x):
        return self.output(torch.relu(self.hidden(x)))

torch.manual_seed(SEED)
model = TinyMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.15, momentum=0.9)
print([(name, tuple(parameter.shape), parameter.is_leaf) for name, parameter in model.named_parameters()])

## 5. Optimizer state is not model parameter state

Plain SGD uses only current gradients. Momentum SGD additionally stores a
**momentum buffer** for each parameter; Adam stores running first and second
moment estimates plus a step count. These tensors belong to
`optimizer.state_dict()`, not `model.state_dict()`.

Saving model weights alone can reproduce inference, but resuming the exact
training trajectory also requires optimizer state (and relevant random state).
A learning rate is stored in optimizer **parameter groups**.

**Checkpoint 5A.** Where would you inspect the current learning rate?

**Checkpoint 5B.** Why can two runs starting from identical model weights
diverge after resume if one discards momentum state?


In [ ]:
X_train = torch.tensor([
    [-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]
]).repeat((24, 1))
y_train = torch.tensor([0, 1, 1, 0], dtype=torch.long).repeat(24)

optimizer.zero_grad(set_to_none=True)
logits = model(X_train)
loss = criterion(logits, y_train)
loss.backward()
before = {name: p.detach().clone() for name, p in model.named_parameters()}
optimizer.step()
after = {name: p.detach().clone() for name, p in model.named_parameters()}
movement = {name: torch.linalg.vector_norm(after[name] - before[name]).item() for name in before}
state = optimizer.state_dict()
print("learning rate:", state["param_groups"][0]["lr"])
print("state entries after first momentum step:", len(state["state"]))
print("parameter movement:", movement)
assert len(state["state"]) == len(list(model.parameters()))
assert max(movement.values()) > 0.0

## 6. Deterministic CPU-small training and evaluation

Training mode is selected with `model.train()`. Evaluation uses
`model.eval()` and `torch.no_grad()`. Today the model has no stochastic or
stateful layer, so train/eval outputs happen to match; Session 5 shows why the
mode call is still mandatory.

We reconstruct the model and optimizer under the same seed, train on a fixed
full batch, and assert invariants: finite falling loss, parameter movement, and
correct center decisions. We do not assert exact final weights.

**Checkpoint 6A.** What two independent effects does `torch.no_grad()`
have during evaluation?

**Checkpoint 6B.** Does `model.eval()` disable gradient recording?


In [ ]:
def seeded_training_run():
    torch.manual_seed(SEED)
    net = TinyMLP()
    opt = torch.optim.Adam(net.parameters(), lr=0.04)
    losses = []
    net.train()
    initial = {name: p.detach().clone() for name, p in net.named_parameters()}
    for _ in range(500):
        opt.zero_grad(set_to_none=True)
        batch_logits = net(X_train)
        batch_loss = criterion(batch_logits, y_train)
        batch_loss.backward()
        opt.step()
        losses.append(batch_loss.detach().item())
    net.eval()
    centers = torch.tensor([[-1.0, -1.0], [-1.0, 1.0], [1.0, -1.0], [1.0, 1.0]])
    with torch.no_grad():
        predictions = net(centers).argmax(dim=1)
    moved = max(
        torch.linalg.vector_norm(p.detach() - initial[name]).item()
        for name, p in net.named_parameters()
    )
    return net, torch.tensor(losses), predictions, moved

trained_model, torch_losses, predictions, moved = seeded_training_run()
print("first/final:", torch_losses[0].item(), torch_losses[-1].item())
print("predictions:", predictions.tolist(), "| max movement:", moved)
assert torch.isfinite(torch_losses).all()
assert torch_losses[-1] < torch_losses[0]
assert predictions.tolist() == [0, 1, 1, 0]
assert moved > 0.1

_, repeat_losses, repeat_predictions, _ = seeded_training_run()
assert torch.allclose(torch_losses, repeat_losses, atol=ATOL, rtol=RTOL)
assert torch.equal(predictions, repeat_predictions)

## 7. Worked lifecycle audit, pitfalls, and exam connections

**Worked audit.** A loop is
`zero_grad → logits → loss → step → backward`. On its first iteration,
`step` sees every `.grad is None`, so parameters do not move. Backward then
populates gradients; on the second iteration, `zero_grad` immediately
deletes them. The loop can report losses but performs **zero effective
updates**. Move `backward` before `step`, then certify parameter movement.

Other pitfalls:

- converting `loss` to NumPy before backward detaches it from the graph;
- using `retain_graph=True` to hide a repeated-backward design error wastes
  memory;
- omitting `zero_grad` accidentally accumulates gradients;
- saving only model state is insufficient for exact optimizer resume;
- setting a seed after model construction does not reproduce initialization.

Round 1 questions often ask which line moves weights, which line constructs
gradients, or what survives in each state dictionary. Session 5 adds mode-
dependent layers and buffers; C7 applies this same lifecycle to CNN parameters.

**Checkpoint 7A.** Which line actually mutates the parameters?

**Checkpoint 7B.** A parameter has `.grad is None` after backward. Give one
benign and one suspicious explanation.


## Checkpoint answers

**1A.** `w` is the leaf; `z.grad_fn` records the operation that produced
the nonleaf. **1B.** No; an optimizer step (or explicit update) is required.

**2A.** $(D,N)(N)=(D)$. **2B.** $1/N$.

**3A.** $2g$. **3B.** It must remove gradients left by the preceding
iteration, not the new gradients intended for this step.

**4A.** The newly computed gradients are cleared, so `step` has nothing to
use. **4B.** `no_grad` prevents graph recording but does not forbid an
optimizer from mutating parameters.

**5A.** `optimizer.param_groups` or the `param_groups` section of its state
dictionary. **5B.** The next update depends on the stored velocity, not only
current weights and gradients.

**6A.** It avoids building an autograd graph and reduces memory/work.
**6B.** No; it changes module behavior, while gradient context is controlled
separately.

**7A.** `optimizer.step()`. **7B.** Benign: the parameter was deliberately
unused/frozen; suspicious: the forward path accidentally disconnected it.
